# [Chapter 18 — Dengue, §18.4] Calibrating the Vector-Host Model to Seasonal Dengue Surveillance

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 18 — Dengue
**Considerations developed:** 2 (short-term assumptions), 11 (correct sensitivity), 13 (equilibrium vs invasion)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_18_dengue/01_dengue_seasonal_calibration.ipynb)


## What this notebook does

Loads 5 years of monthly synthetic dengue incidence and fits a vector-host SIRS model with seasonal vector mortality. The notebook shows that a single average $\mathcal{R}_0$ misrepresents the dynamics, while reporting both the wet-season and dry-season $\mathcal{R}_0$ envelope captures the operationally important variation.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_18
from shared.verification import assert_within_tolerance
set_seed_chapter_18()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Load data

In [ ]:
with open(os.path.join(DATA_ROOT, 'dengue', 'monthly_cases_per_100k.csv')) as f:
    next(f)
    rows = [(int(m), int(c)) for m, c in csv.reader(f)]
month = np.array([r[0] for r in rows])
cases = np.array([r[1] for r in rows], dtype=float)
with open(os.path.join(DATA_ROOT, 'dengue', 'metadata.json')) as f:
    truth = json.load(f)['generating_parameters_TRUTH_FOR_NOTEBOOK_VERIFICATION']
print(f'Data: {len(month)} months')
print(f'Annual peak months 1, 23, 35, 47, 59 (~12-month period)')


## Visualize the seasonality

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(month, cases, 'o-', color=BOOK_COLORS['infectious'], lw=1.5, ms=4)
ax.set_xlabel('month index (1 = year 1 month 1)')
ax.set_ylabel('cases per 100K')
ax.set_title('5 years of monthly dengue surveillance')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Fit the seasonal R_0(t) envelope

We fit two parameters: the baseline vector-to-host ratio $\overline{M/N}$ and the seasonal amplitude $A$ that modulates vector mortality. The other Ross-Macdonald parameters $(a, b, c, \gamma_h)$ are fixed at literature values (the notebook treats $a, b, c, \gamma_h$ as known and concentrates on the parameters most sensitive to local context).

In [ ]:
from shared.parameters import baseline_chapter_18
p = baseline_chapter_18()
a, b, c, gamma_h, mu_v_baseline, xi_h = (
    p['a'], p['b'], p['c'], p['gamma_h'],
    p['mu_v_baseline'], 1.0/(2.0*365.0))

def vh_model(MN_bar, A_amp, t_days):
    def rhs(y, t):
        SH, IH, RH, SV, IV = y
        mu_v = mu_v_baseline * (1 + A_amp * np.cos(2*np.pi*t/365 + np.pi))
        Lam_h = a * b * MN_bar * IV
        Lam_v = a * c * IH
        return [-Lam_h*SH + xi_h*RH,
                Lam_h*SH - gamma_h*IH,
                gamma_h*IH - xi_h*RH,
                mu_v*(1-SV) - Lam_v*SV,
                Lam_v*SV - mu_v*IV]
    y0 = [0.95, 0.001, 0.049, 1.0, 0.0]
    sol = odeint(rhs, y0, t_days)
    return sol

def monthly_predicted(MN_bar, A_amp):
    t_days = np.linspace(0, 365*5, 365*5+1)
    sol = vh_model(MN_bar, A_amp, t_days)
    SH, IH, RH, SV, IV = sol.T
    new_inf = np.maximum(a * b * MN_bar * IV * SH, 0.0)
    monthly = []
    for i in range(60):
        lo, hi = i*30, min((i+1)*30, len(new_inf))
        monthly.append(new_inf[lo:hi].sum() * 100_000)
    return np.array(monthly)

def loss(theta):
    MN_bar, A_amp = theta
    if MN_bar <= 0 or A_amp <= 0 or A_amp >= 1: return 1e20
    pred = monthly_predicted(MN_bar, A_amp)
    # Fit later months to avoid initial-transient dependence
    return float(np.sum((np.log1p(pred[12:]) - np.log1p(cases[12:]))**2))

res = minimize(loss, x0=[4.0, 0.4], method='Nelder-Mead')
MN_hat, A_hat = float(res.x[0]), float(res.x[1])
print(f'Estimated  M/N_bar = {MN_hat:.2f}  (truth: {truth["M_over_N"]})')
print(f'Estimated  A_amp   = {A_hat:.2f}  (truth: {truth["mu_v_amplitude"]})')


## R_0(t) envelope from fitted parameters

In [ ]:
import math
t_year = np.linspace(0, 365, 365)
mu_v_t = mu_v_baseline * (1 + A_hat * np.cos(2*np.pi*t_year/365 + np.pi))
R0_t = np.sqrt(a**2 * b * c * MN_hat / (mu_v_t * gamma_h))
R0_min, R0_max = R0_t.min(), R0_t.max()
R0_mean = R0_t.mean()
print(f'\nFitted R_0(t) envelope:')
print(f'  Wet-season peak  R_0 = {R0_max:.2f}')
print(f'  Dry-season floor R_0 = {R0_min:.2f}')
print(f'  Annual mean      R_0 = {R0_mean:.2f}')
print(f'\nReporting only the mean would obscure {(R0_max-R0_min):.2f} units of seasonal variation.')

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(t_year/30, R0_t, color=BOOK_COLORS['primary'], lw=2)
ax.axhline(R0_mean, ls='--', color=BOOK_COLORS['highlight'], label=f'annual mean R_0 = {R0_mean:.2f}')
ax.axhline(1.0, ls=':', color='gray')
ax.set_xlabel('month of year')
ax.set_ylabel('R_0(t)')
ax.set_title('Fitted seasonal R_0(t) — single-number reporting hides this signal (Consideration 13)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verify: amplitude estimate within 50% of truth
assert abs(A_hat - truth['mu_v_amplitude']) / truth['mu_v_amplitude'] < 0.5, 'A estimate too far from truth'
print('\nVerified: seasonal amplitude recovered within 50% of truth.')


## What's next

- Notebook 02 sweeps vector-control intervention scenarios using sensitivity analysis to identify the most cost-effective lever (vector reduction vs biting-rate reduction vs human-immunity boost).
- Real dengue fits use environmental covariates (precipitation, temperature) for the seasonal forcing; this notebook's sinusoid is a useful starting point but a coarse approximation.